In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\data\Telco-Customer-Churn.csv')

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [3]:
df = df.drop("customerID", axis=1)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df_encoded = pd.get_dummies(df, drop_first=True)

In [ ]:
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]


### Logistic Regression

In [5]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline_lr.fit(X_train, y_train)

y_pred_lr = pipeline_lr.predict(X_test)
y_prob_lr = pipeline_lr.predict_proba(X_test)[:, 1]

#Training Accurcy
training_accuracy_lr = accuracy_score(y_train, pipeline_lr.predict(X_train))

#Testing Accuracy
testing_accuracy_lr = accuracy_score(y_test, y_pred_lr)

# Validation (cross-validated) accuracy and ROC AUC on training data
cv_scores_lr = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring='accuracy')
cv_roc_auc_lr = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring='roc_auc')

print("Logistic Regression Model Performance:")
print(f"Training Accuracy: {training_accuracy_lr:.4f}")
print(f"Testing Accuracy: {testing_accuracy_lr:.4f}")
print(f"Cross-Validated Accuracy: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std() * 2:.4f})")
print(f"Cross-Validated ROC AUC: {cv_roc_auc_lr.mean():.4f} (+/- {cv_roc_auc_lr.std() * 2:.4f})")


Logistic Regression Model Performance:
Training Accuracy: 0.8028
Testing Accuracy: 0.8204
Cross-Validated Accuracy: 0.8012 (+/- 0.0146)
Cross-Validated ROC AUC: 0.8408 (+/- 0.0235)


### Gradient Boosting

In [6]:
# Gradient Boosting classifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

pipeline_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('gb', GradientBoostingClassifier(random_state=42))
])

pipeline_gb.fit(X_train, y_train)

# Predictions and probabilities
y_pred_gb = pipeline_gb.predict(X_test)
y_proba_gb = pipeline_gb.predict_proba(X_test)[:, 1]

# Training accuracy
train_acc = accuracy_score(y_train, pipeline_gb.predict(X_train))

# Validation (cross-validated) accuracy and ROC AUC on training data
cv_scores_acc = cross_val_score(pipeline_gb, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
cv_scores_gb = cross_val_score(pipeline_gb, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)

test_acc_gb = accuracy_score(y_test, y_pred_gb)

print("Gradient Boosting Model Performance:")
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy: {test_acc_gb:.4f}")
print(f"Cross-Validated Accuracy: {cv_scores_acc.mean():.4f} (+/- {cv_scores_acc.std() * 2:.4f})")
print(f"Cross-Validated ROC AUC: {cv_scores_gb.mean():.4f} (+/- {cv_scores_gb.std() * 2:.4f})")

Gradient Boosting Model Performance:
Training Accuracy: 0.8257
Testing Accuracy: 0.8070
Cross-Validated Accuracy: 0.8000 (+/- 0.0220)
Cross-Validated ROC AUC: 0.8415 (+/- 0.0190)


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
model_probabilities = {
    "Logistic Regression": y_prob_lr,
    "Gradient Boosting": y_proba_gb
}

threshold_results = []

for model_name, probabilities in model_probabilities.items():
    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)
        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(y_test, predictions, zero_division=0),
            "Recall": recall_score(y_test, predictions, zero_division=0),
            "F1 Score": f1_score(y_test, predictions, zero_division=0)
        })

threshold_results_df = pd.DataFrame(threshold_results)

print("Threshold Comparison:")
print(threshold_results_df.round(4).to_string(index=False))

Threshold Comparison:
              Model  Threshold  Accuracy  Precision  Recall  F1 Score
Logistic Regression       0.30    0.7764     0.5537  0.8016    0.6550
Logistic Regression       0.35    0.7842     0.5714  0.7399    0.6449
Logistic Regression       0.40    0.7970     0.6000  0.6997    0.6460
Logistic Regression       0.45    0.8119     0.6459  0.6408    0.6433
Logistic Regression       0.50    0.8204     0.6852  0.5952    0.6370
Logistic Regression       0.55    0.8169     0.7138  0.5147    0.5981
Logistic Regression       0.60    0.8084     0.7464  0.4182    0.5361
  Gradient Boosting       0.30    0.7771     0.5553  0.7936    0.6534
  Gradient Boosting       0.35    0.8041     0.6048  0.7507    0.6699
  Gradient Boosting       0.40    0.8141     0.6377  0.6890    0.6624
  Gradient Boosting       0.45    0.8162     0.6629  0.6220    0.6418
  Gradient Boosting       0.50    0.8070     0.6667  0.5416    0.5976
  Gradient Boosting       0.55    0.8034     0.6951  0.4584    0.552